In [1]:
# ** This cell is needed since we are not in the src directory 
import sys 
import os
# Add the src/ directory to the Python path
sys.path.append(os.path.abspath(os.path.join(os.getcwd(), "../src/")))

ROOT_DIR = ".."
SRC_DIR = ROOT_DIR + "/src"

import sys
sys.path.append("/Users/admin/eeg-ds004504/")


In [2]:
from config_handler import initiate_config, load_config

In [3]:
initiate_config()

{'data_path': '/Users/admin/eeg-ds004504',
 'derivatives': False,
 'freqBands': {'Alpha': [8, 12],
  'Beta': [12, 30],
  'Delta': [0.5, 4],
  'Theta': [4, 8],
  'custom1': [9, 11]},
 'method': 'welch',
 'stepSize': 0.3,
 'windowLength': 3}

In [4]:
print(load_config())

{'data_path': '/Users/admin/eeg-ds004504', 'derivatives': False, 'freqBands': {'Alpha': [8, 12], 'Beta': [12, 30], 'Delta': [0.5, 4], 'Theta': [4, 8], 'custom1': [9, 11]}, 'method': 'welch', 'stepSize': 0.3, 'windowLength': 3}


In [5]:
from pyspark.sql import SparkSession
from pyspark.sql.functions import pandas_udf, PandasUDFType
from pyspark.sql.types import StructType, StructField, StringType, IntegerType, DoubleType, ArrayType, MapType
import pandas as pd

In [6]:
# Check if there's an active Spark context and stop it
from pyspark import SparkContext
if SparkContext._active_spark_context:
    print("Stopping existing Spark context...")
    SparkContext._active_spark_context.stop()
    print("Previous Spark context stopped successfully")

In [7]:
import os
from pyspark.sql import SparkSession

# Set Java options for the JVM running Spark
# -Xmx12g : Sets the maximum heap size to 12GB
# -Xms4g : Sets the initial heap size to 4GB to avoid resizing overhead
os.environ["_JAVA_OPTIONS"] = "-Xmx12g -Xms4g"

# Build Spark session with memory, parallelism, and network settings
spark = (
    SparkSession.builder 
    # Application name shown in Spark UI
    .appName("EEG_Analysis") 

    # Use all available logical cores or specify a number
    # "local[*]" uses all available cores, "local[12]" limits to 12 threads
    .config("spark.master", "local[12]") \

    # Executor memory: how much memory each Spark worker can use
    .config("spark.executor.memory", "8g") \

    # Driver memory: memory available to the Spark driver (main Python process)
    .config("spark.driver.memory", "8g") \

    # Number of shuffle partitions (e.g., after groupBy, join, etc.)
    # Lower this in local mode to reduce overhead (default is 200)
    .config("spark.sql.shuffle.partitions", "12") \

    # Default number of partitions in operations like parallelize
    .config("spark.default.parallelism", "12") \

    # Maximum size (in MB) allowed for any RPC message (e.g., large UDF closures or data broadcasts)
    .config("spark.rpc.message.maxSize", "256") \

    # Required for avoiding binding issues on some MacOS environments
    .config("spark.driver.bindAddress", "127.0.0.1") \
    .config("spark.driver.host", "127.0.0.1") 
    .getOrCreate()
)

    # ----------------------------------------
    # Additional advanced options (optional):
    # ----------------------------------------

    # Use Kryo serializer instead of default Java serializer for better performance
    # .config("spark.serializer", "org.apache.spark.serializer.KryoSerializer") \

    # Increase broadcast join timeout (in seconds) for large models or lookup tables
    # .config("spark.sql.broadcastTimeout", "600") \

    # Fraction of JVM memory reserved for execution and storage (default is 0.6)
    # .config("spark.memory.fraction", "0.8") \

    # Portion of memory reserved for caching/storage (default is 0.5 of memory.fraction)
    # .config("spark.memory.storageFraction", "0.3") \

    # Enable Apache Arrow for efficient pandas-to-Spark conversion (useful with UDFs)
    # .config("spark.sql.execution.arrow.pyspark.enabled", "true") \

    # Finalize and create the Spark session


# spark = SparkSession.builder.appName("MyApp").getOrCreate()

print("New Spark session created successfully")

Picked up _JAVA_OPTIONS: -Xmx12g -Xms4g
Picked up _JAVA_OPTIONS: -Xmx12g -Xms4g
Setting default log level to "WARN".
To adjust logging level use sc.setLogLevel(newLevel). For SparkR, use setLogLevel(newLevel).
25/04/18 22:19:33 WARN NativeCodeLoader: Unable to load native-hadoop library for your platform... using builtin-java classes where applicable


New Spark session created successfully


In [8]:
from populate_schemas import load_subjects_df, extract_features_udtf
from feature_extraction import processEpoch, processSub
from schema_definition import get_feature_schema, get_subject_schema

sc = spark.sparkContext # we pass udf/udtf's (user defind functions and user defined table functions) to spark so it can access them

# Making all necessary modules available to spark
try:
    sc.addPyFile(os.path.join(SRC_DIR, "feature_extraction.py"))
    print("Added feature_extraction.py to the pyspark context")
    sc.addPyFile(os.path.join(SRC_DIR, "preprocess_sets.py"))
    print("Added preprocess_sets to the pyspark context")
    sc.addPyFile(os.path.join(SRC_DIR, "schema_definition.py"))
    print("Added schema_definition.py to the pyspark context")
    sc.addPyFile(os.path.join(SRC_DIR, "config_handler.py"))
    print("Added config_handler.py to the pyspark context")
except Exception as e:
    print(f"Error adding files to SparkContext: {e}")

Config not found in feature_extraction.py
Config using in feature Extraction.py {'data_path': '/Users/admin/eeg-ds004504', 'derivatives': False, 'freqBands': {'Alpha': [8, 12], 'Beta': [12, 30], 'Delta': [0.5, 4], 'Theta': [4, 8], 'custom1': [9, 11]}, 'method': 'welch', 'stepSize': 0.3, 'windowLength': 3}
Config using in feature Extraction.py {'data_path': '/Users/admin/eeg-ds004504', 'derivatives': False, 'freqBands': {'Alpha': [8, 12], 'Beta': [12, 30], 'Delta': [0.5, 4], 'Theta': [4, 8], 'custom1': [9, 11]}, 'method': 'welch', 'stepSize': 0.3, 'windowLength': 3}
Added feature_extraction.py to the pyspark context
Added preprocess_sets to the pyspark context
Added schema_definition.py to the pyspark context
Added config_handler.py to the pyspark context


In [9]:
subject_df = load_subjects_df(spark) #this is the .tsv with the information of all the participants

In [10]:
# creating table for sub1 then saving it as a .pkl

In [11]:
%%time
# we need to give the path of our  data directory to process the EEG data from
from preprocess_sets import get_data_path

# set_data_path("/Users/user/eeg-ds004504") !!! this doesn't work! so we need to do it manually in preprocess_sets.py! or else won't work!
print(get_data_path())

#Example below is how to get a single subject  and extract its features
sub1 = (
    subject_df
    .filter((subject_df.SubjectID == "sub-001"))
    .groupBy("SubjectID")
    .apply(extract_features_udtf)
)
print("3 tables types")
sub1.filter((sub1.table_type=="band")).show(3)
sub1.filter((sub1.table_type=="electrode")).show(3)
sub1.filter((sub1.table_type=="epoch")).show(3)

/Users/admin/eeg-ds004504
3 tables types


/Users/admin/neuro-venv/lib/python3.9/site-packages/pyspark/sql/pandas/group_ops.py:104: UserWarning: It is preferred to use 'applyInPandas' over this API. This API will be deprecated in the future releases. See SPARK-28264 for more details.
  warnings.warn(
Config not found in feature_extraction.py                           (0 + 1) / 1]
Config using in feature Extraction.py {'data_path': '/Users/admin/eeg-ds004504', 'derivatives': False, 'freqBands': {'Alpha': [8, 12], 'Beta': [12, 30], 'Delta': [0.5, 4], 'Theta': [4, 8], 'custom1': [9, 11]}, 'method': 'welch', 'stepSize': 0.3, 'windowLength': 3}
processSub sub-001
subPath sub-001
subPath data_path /Users/admin/eeg-ds004504
Path handed: /Users/admin/eeg-ds004504/ds004504/sub-001/eeg/sub-001_task-eyesclosed_eeg.set
                                                                                

+---------+-------+---------+--------+-----------+------------+----------+
|SubjectID|EpochID|Electrode|WaveBand|FeatureName|FeatureValue|table_type|
+---------+-------+---------+--------+-----------+------------+----------+
|  sub-001|   ep-0|      Fp1|   Alpha|      Power|0.0011432203|      band|
|  sub-001|   ep-0|      Fp1|    Beta|      Power|1.9580401E-4|      band|
|  sub-001|   ep-0|      Fp1|   Delta|      Power|  0.08257893|      band|
+---------+-------+---------+--------+-----------+------------+----------+
only showing top 3 rows



Config not found in feature_extraction.py                           (0 + 1) / 1]
Config using in feature Extraction.py {'data_path': '/Users/admin/eeg-ds004504', 'derivatives': False, 'freqBands': {'Alpha': [8, 12], 'Beta': [12, 30], 'Delta': [0.5, 4], 'Theta': [4, 8], 'custom1': [9, 11]}, 'method': 'welch', 'stepSize': 0.3, 'windowLength': 3}
processSub sub-001
subPath sub-001
subPath data_path /Users/admin/eeg-ds004504
Path handed: /Users/admin/eeg-ds004504/ds004504/sub-001/eeg/sub-001_task-eyesclosed_eeg.set
                                                                                

+---------+-------+---------+--------+-----------+------------+----------+
|SubjectID|EpochID|Electrode|WaveBand|FeatureName|FeatureValue|table_type|
+---------+-------+---------+--------+-----------+------------+----------+
|  sub-001|   ep-0|      Fp1|    NULL|TotalEnergy|  0.24224812| electrode|
|  sub-001|   ep-0|      Fp1|    NULL| TotalPower| 0.011235955| electrode|
|  sub-001|   ep-0|      Fp2|    NULL|TotalEnergy|  0.24001254| electrode|
+---------+-------+---------+--------+-----------+------------+----------+
only showing top 3 rows



Config not found in feature_extraction.py                           (0 + 1) / 1]
Config using in feature Extraction.py {'data_path': '/Users/admin/eeg-ds004504', 'derivatives': True, 'freqBands': {'Alpha': [8, 12], 'Beta': [12, 30], 'Delta': [0.5, 4], 'Theta': [4, 8], 'custom1': [9, 11]}, 'method': 'welch', 'stepSize': 0.3, 'windowLength': 3}
processSub sub-001
subPath sub-001
subPath data_path /Users/admin/eeg-ds004504
Path handed: /Users/admin/eeg-ds004504/ds004504/sub-001/eeg/sub-001_task-eyesclosed_eeg.set
[Stage 8:>                                                          (0 + 1) / 1]

+---------+-------+---------+--------+-----------+-------------+----------+
|SubjectID|EpochID|Electrode|WaveBand|FeatureName| FeatureValue|table_type|
+---------+-------+---------+--------+-----------+-------------+----------+
|  sub-001|   ep-0|     NULL|    NULL|       Mean|2.0468264E-20|     epoch|
|  sub-001|   ep-0|     NULL|    NULL|        Std| 4.6993853E-5|     epoch|
|  sub-001|   ep-0|     NULL|    NULL|   Variance| 2.3622075E-9|     epoch|
+---------+-------+---------+--------+-----------+-------------+----------+
only showing top 3 rows

CPU times: user 156 ms, sys: 101 ms, total: 258 ms
Wall time: 32min 21s


In [12]:
sub1.printSchema()

root
 |-- SubjectID: string (nullable = false)
 |-- EpochID: string (nullable = false)
 |-- Electrode: string (nullable = true)
 |-- WaveBand: string (nullable = true)
 |-- FeatureName: string (nullable = true)
 |-- FeatureValue: float (nullable = true)
 |-- table_type: string (nullable = true)



In [13]:
sub1.show()

Config not found in feature_extraction.py                           (0 + 1) / 1]
Config using in feature Extraction.py {'data_path': '/Users/admin/eeg-ds004504', 'derivatives': True, 'freqBands': {'Alpha': [8, 12], 'Beta': [12, 30], 'Delta': [0.5, 4], 'Theta': [4, 8], 'custom1': [9, 11]}, 'method': 'welch', 'stepSize': 0.3, 'windowLength': 3}
processSub sub-001
subPath sub-001
subPath data_path /Users/admin/eeg-ds004504
Path handed: /Users/admin/eeg-ds004504/ds004504/sub-001/eeg/sub-001_task-eyesclosed_eeg.set
[Stage 11:>                                                         (0 + 1) / 1]

+---------+-------+---------+--------+-----------+------------+----------+
|SubjectID|EpochID|Electrode|WaveBand|FeatureName|FeatureValue|table_type|
+---------+-------+---------+--------+-----------+------------+----------+
|  sub-001|   ep-0|      Fp1|   Alpha|      Power|0.0011432203|      band|
|  sub-001|   ep-0|      Fp1|    Beta|      Power|1.9580401E-4|      band|
|  sub-001|   ep-0|      Fp1|   Delta|      Power|  0.08257893|      band|
|  sub-001|   ep-0|      Fp1|   Theta|      Power|0.0056116455|      band|
|  sub-001|   ep-0|      Fp1| custom1|      Power|0.0010104624|      band|
|  sub-001|   ep-0|      Fp1|    NULL|TotalEnergy|  0.24224812| electrode|
|  sub-001|   ep-0|      Fp1|    NULL| TotalPower| 0.011235955| electrode|
|  sub-001|   ep-0|      Fp2|   Alpha|      Power| 9.638189E-4|      band|
|  sub-001|   ep-0|      Fp2|    Beta|      Power|1.7688205E-4|      band|
|  sub-001|   ep-0|      Fp2|   Delta|      Power| 0.083892204|      band|
|  sub-001|   ep-0|      

In [ ]:
from datetime import import datetime
import os
from config_handler import load_config

def create_database_log(df, filename, config, timestamp):
    """
    Creates a log file with information about the saved dataframe.
    
    Parameters:
    - df: The Spark DataFrame being saved
    - filename: The name of the saved pkl file
    - config: Configuration dictionary from load_config()
    """
    # Count unique subjects
    try:
        unique_subjects = df.select("SubjectID").distinct().count()
    except:
        unique_subjects = -1
    
    # Create log entry
    log_entry = [
        f"=== DATABASE LOG ENTRY: {timestamp} ===",
        f"Saved file: {filename}",
        f"Unique subjects: {unique_subjects}",
        "\nConfiguration:",
    ]
    
    # Write to log file
    with open("databaselog.txt", "a") as f:
        f.write("\n".join(log_entry))
    
    print(f"Log entry written to databaselog.txt")

In [ ]:
from datetime import import datetime

# Generate a timestamp like "Apr14_2230"
timestamp = datetime.now().strftime("%b%d_%H%M")
# **** I THINK SHOULD PIVOT for all 3 tables, GROUP BY SUBJECTID, EPOCHID and then save so that the new one will match
# Save with timestamp in filename
pandas_df = sub1.toPandas()
filename = f"sub1_old_{timestamp}.pkl"
pandas_df.to_pickle(filename)

# Get the configuration
config = load_config()

# Create log entry
create_database_log(sub1, filename, config, timestamp)